In [ ]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [3]:
# Настройки
SOURCE_REPO_ID = "DeepPavlov/clarqa"
LOCAL_SAVE_PATH = "./clarqa_ru"
CACHE_FILE = "translation_cache.jsonl"

# Конфигурации и сплиты
CONFIGS = ['single_turn', 'multi_turn']
SPLITS = ['train', 'dev', 'test']

In [4]:
def load_cache() -> dict[str, str]:
    """Загружает кэш переводов из файла"""
    cache: dict[str, str] = {}
    if not os.path.exists(CACHE_FILE):
        return cache
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj: dict[str, str] = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except Exception:
                    continue
    except Exception as e:
        logging.error(f"Failed to load cache: {e}")
    return cache

def append_cache(text: str, translation: str) -> None:
    """Сохраняет перевод в кэш"""
    try:
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")
    except Exception as e:
        logging.error(f"Failed to append cache: {e}")

translation_cache: dict[str, str] = load_cache()

In [5]:
def translate_with_yandex(text: str, retries: int = 3, delay: int = 5) -> tuple[str, bool]:
    """
    Переводит текст через Yandex Translate с использованием кэша
    Возвращает (перевод, успех_или_нет)
    """
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    
    # Проверяем кэш
    if text in translation_cache:
        return translation_cache[text], True
    
    for attempt in range(retries):
        try:
            time.sleep(0.5)
            result = yandex.translate(text, "ru")
            
            if hasattr(result, 'result'):
                translated_text = str(result.result)
            else:
                translated_text = str(result)
            
            # Сохраняем в кэш
            translation_cache[text] = translated_text
            append_cache(text, translated_text)
            return translated_text, True
            
        except Exception as e:
            logging.warning(f"Translation error: '{text[:50]}...'. Attempt {attempt + 1}/{retries}. Error: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    
    logging.error(f"Failed to translate: '{text[:50]}...'")
    return "", False

In [16]:
def translate_context(context_field):
    """
    Переводит поле context, которое может быть:
    - списком строк (как в большинстве случаев)
    - обычной строкой
    - или другим типом
    """
    # Если это список - переводим каждый элемент
    if isinstance(context_field, list):
        translated_context = []
        all_success = True
        
        for text in context_field:
            if isinstance(text, str) and text.strip():
                translated_text, success = translate_with_yandex(text)
                translated_context.append(translated_text)
                if not success:
                    all_success = False
            else:
                translated_context.append(text)
                all_success = False
        
        return translated_context, all_success
    
    # Если это строка - переводим целиком (НЕ разбиваем на символы!)
    elif isinstance(context_field, str):
        translated_text, success = translate_with_yandex(context_field)
        return translated_text, success
    
    # Если что-то другое - возвращаем как есть
    else:
        return context_field, True

In [17]:
def translate_entity(entity_str: str) -> tuple[str, bool]:
    """
    Переводит entity строку формата:
    "имя <S> метки <S> описание"
    
    Переводит:
    - имя (до первого <S>)
    - описание (после последнего <S>)
    
    Оставляет без изменений:
    - метки (между <S> и <S>)
    """
    if not isinstance(entity_str, str) or entity_str.strip() == "":
        return "", True
    
    # Ищем все разделители
    parts = entity_str.split("<S>")
    
    if len(parts) >= 3:
        # parts[0] - имя сущности
        # parts[1] - метки (не переводим)
        # parts[2:] - описание (может содержать <S> внутри)
        
        name = parts[0].strip()
        categories = parts[1].strip() if len(parts) > 1 else ""
        description = "<S>".join(parts[2:]).strip() if len(parts) > 2 else ""
        
        # Переводим имя
        translated_name, success1 = translate_with_yandex(name)
        
        # Переводим описание (если есть)
        translated_description, success2 = ("", True)
        if description:
            translated_description, success2 = translate_with_yandex(description)
        
        success = success1 and success2
        
        # Собираем обратно
        result_parts = [translated_name, categories]
        if translated_description:
            result_parts.append(translated_description)
        
        return " <S> ".join(result_parts), success
    
    elif len(parts) == 2:
        # Только имя и метки (нет описания)
        name = parts[0].strip()
        categories = parts[1].strip()
        translated_name, success = translate_with_yandex(name)
        return f"{translated_name} <S> {categories}", success
    else:
        # Нет разделителей — переводим всё
        return translate_with_yandex(entity_str)

In [20]:
def process_and_translate_split(split_name, source_dataset_split, progress_file):
    """
    Обрабатывает и переводит один сплит датасета
    """
    translated_records = []
    failed_indices = []
    
    # Загружаем уже переведенные записи
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed_indices.append(record.get('_index', -1))
                    else:
                        translated_records.append(record)
                except Exception:
                    continue
        logging.info(f"[{split_name}] Resuming. Found {len(translated_records)} successful, {len(failed_indices)} failed records.")
    
    successful_indices = {record['_index'] for record in translated_records if '_index' in record}
    start_index = len(translated_records) + len(failed_indices)
    total_records = len(source_dataset_split)
    
    if start_index < total_records:
        logging.info(f"[{split_name}] Starting translation from index {start_index}...")
        
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_dataset_split.select(range(start_index, total_records))),
                desc=f"Translating {split_name}",
                total=total_records - start_index
            )
            
            for idx, example in pbar:
                global_idx = start_index + idx
                
                if global_idx in successful_indices:
                    continue
                
                # 1. Переводим context (может быть списком или строкой)
                original_context = example['context']
                translated_context, context_success = translate_context(original_context)
                
                # 2. Переводим entity1
                original_entity1 = example.get('entity1', '')
                translated_entity1, entity1_success = translate_entity(original_entity1)
                
                # 3. Переводим entity2
                original_entity2 = example.get('entity2', '')
                translated_entity2, entity2_success = translate_entity(original_entity2)
                
                # Создаём запись
                new_record = {
                    '_index': global_idx,
                    '_failed': not (context_success and entity1_success and entity2_success),
                    'context': original_context,
                    'context_ru': translated_context,
                    'entity1': original_entity1,
                    'entity1_ru': translated_entity1,
                    'entity2': original_entity2,
                    'entity2_ru': translated_entity2,
                    'label': example.get('label', 0)
                }
                
                f.write(json.dumps(new_record, ensure_ascii=False) + "\n")
                f.flush()

                time.sleep(0.5)
                
                if not new_record['_failed']:
                    translated_records.append(new_record)
                else:
                    failed_indices.append(global_idx)
    
    # Загружаем все успешные записи
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if not record.get('_failed', False):
                        record.pop('_index', None)
                        record.pop('_failed', None)
                        all_successful.append(record)
                except Exception:
                    continue
    
    if not all_successful:
        logging.error(f"[{split_name}] No records were successfully translated. Aborting.")
        return None
    
    total = len(source_dataset_split)
    successful_count = len(all_successful)
    failed_count = total - successful_count
    logging.info(f"[{split_name}] Translation completed: {successful_count}/{total} successful ({failed_count} failed)")
    
    if failed_count > 0:
        logging.info(f"[{split_name}] To retry failed translations, delete or modify {progress_file} and run again")
    
    return Dataset.from_list(all_successful)

In [ ]:
# Загружаем датасет ClarQa и проверяем структуру
logging.info(f"Loading source dataset '{SOURCE_REPO_ID}'...")

# Сначала посмотрим все доступные конфигурации
from datasets import get_dataset_config_names
available_configs = get_dataset_config_names(SOURCE_REPO_ID)
print(f"Available configs: {available_configs}")

# Загружаем каждую конфигурацию для проверки структуры
for config in available_configs:
    print(f"\n{'='*60}")
    print(f"Config: {config}")
    print(f"{'='*60}")
    temp_ds = load_dataset(SOURCE_REPO_ID, config, split='train')
    print(f"Features: {temp_ds.column_names}")
    print(f"Example: {temp_ds[0]}")

In [ ]:
# Переводим все конфигурации и сплиты
print("\n" + "="*60)
print("STARTING TRANSLATION OF ALL CONFIGS AND SPLITS")
print("="*60)

all_configs = {}

for config_name in CONFIGS:
    logging.info(f"\n{'='*60}")
    logging.info(f"Loading config: {config_name}")
    logging.info(f"{'='*60}")
    
    # Загружаем все сплиты для текущей конфигурации
    source_config = load_dataset(SOURCE_REPO_ID, config_name)
    
    translated_splits = {}
    
    for split in SPLITS:
        if split not in source_config:
            logging.warning(f"[{config_name}/{split}] Split not found, skipping...")
            continue
            
        progress_file = f"translated_clarqa_{config_name}_{split}.jsonl"
        
        translated_splits[split] = process_and_translate_split(
            f"{config_name}/{split}",
            source_config[split],
            progress_file
        )
        
        if translated_splits[split] is None:
            logging.error(f"Failed to translate {config_name}/{split}")
            break
    
    # Если все сплиты переведены успешно, сохраняем конфигурацию
    if all(v is not None for v in translated_splits.values()):
        all_configs[config_name] = DatasetDict(translated_splits)
    else:
        logging.error(f"Failed to translate config {config_name}")

In [ ]:
# Сохраняем всё вместе
if all_configs:
    print("\n" + "="*60)
    print("FINAL TRANSLATED DATASET")
    print("="*60)
    
    for config_name, ds_dict in all_configs.items():
        print(f"\n{config_name.upper()}:")
        for split_name, ds in ds_dict.items():
            print(f"  {split_name}: {len(ds)} examples")
            print(f"    Features: {ds.column_names}")
    
    print("\n" + "="*60)
    print("EXAMPLE TRANSLATION")
    print("="*60)
    
    # Показываем пример для первой конфигурации и сплита
    first_config = list(all_configs.keys())[0]
    first_split = list(all_configs[first_config].keys())[0]
    example = all_configs[first_config][first_split][0]
    
    print(f"\nConfig: {first_config}, Split: {first_split}")
    print("\n--- CONTEXT ---")
    print(f"Original: {example['context']}")
    print(f"Translated: {example['context_ru']}")
    
    print("\n--- ENTITY1 ---")
    print(f"Original: {example['entity1'][:300]}...")
    print(f"Translated: {example['entity1_ru'][:300]}...")
    
    print(f"\n--- LABEL ---")
    print(f"label: {example['label']}")
    
    # Сохраняем локально
    final_dataset = DatasetDict(all_configs)
    logging.info(f"\nSaving translated dataset locally to '{LOCAL_SAVE_PATH}'...")
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nDataset saved to {LOCAL_SAVE_PATH}")
else:
    logging.error("No configs were successfully translated. Halting.")

In [ ]:
def check_translation_status():
    """Проверяет статус всех переводов по всем конфигурациям и сплитам"""
    print("="*60)
    print("TRANSLATION STATUS REPORT")
    print("="*60)
    
    total_successful = 0
    total_failed = 0
    
    for config_name in CONFIGS:
        print(f"\n📁 CONFIG: {config_name.upper()}")
        print("-"*40)
        
        for split in SPLITS:
            progress_file = f"translated_clarqa_{config_name}_{split}.jsonl"
            if os.path.exists(progress_file):
                successful = 0
                failed = 0
                with open(progress_file, "r", encoding="utf-8") as f:
                    for line in f:
                        record = json.loads(line)
                        if record.get('_failed', False):
                            failed += 1
                        else:
                            successful += 1
                
                total_successful += successful
                total_failed += failed
                total = successful + failed
                progress = (successful / total * 100) if total > 0 else 0
                print(f"  {split}: {successful} | {failed} | 📊 {progress:.1f}% ({successful}/{total})")
            else:
                print(f"  {split}: No progress file found")
    
    print(f"\n{'='*60}")
    print(f"TOTAL: {total_successful} | {total_failed} | 📊 {total_successful + total_failed} total")
    print(f"{'='*60}")

check_translation_status()

In [ ]:
# ДИАГНОСТИКА И ВОССТАНОВЛЕНИЕ ДАТАСЕТА

import os
import json
from datasets import Dataset, DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")  # Замените на ваш токен

LOCAL_PATH = "/Users/polinakremneva/code/deep-pavlov/clarqa_ru_complete"
REPO_ID = "DeepPavlov/clarqa_ru"

print("="*60)
print("ДИАГНОСТИКА СТРУКТУРЫ ПАПКИ")
print("="*60)

# Исследуем структуру папки
def explore_directory(path, indent=0):
    """Рекурсивно исследуем структуру директории"""
    try:
        items = os.listdir(path)
        for item in sorted(items):
            if item.startswith('.'):
                continue
            full_path = os.path.join(path, item)
            if os.path.isdir(full_path):
                print("  " * indent + f"📁 {item}/")
                explore_directory(full_path, indent + 1)
            else:
                size = os.path.getsize(full_path)
                print("  " * indent + f"📄 {item} ({size} bytes)")
    except PermissionError:
        pass

explore_directory(LOCAL_PATH)

print("\n" + "="*60)
print("ПОИСК ФАЙЛОВ ДАННЫХ")
print("="*60)

# Ищем все .arrow файлы (основные данные датасета)
arrow_files = []
for root, dirs, files in os.walk(LOCAL_PATH):
    for file in files:
        if file.endswith('.arrow'):
            arrow_files.append(os.path.join(root, file))
            print(f"Найден .arrow файл: {os.path.relpath(os.path.join(root, file), LOCAL_PATH)}")

# Ищем dataset_info.json
info_files = []
for root, dirs, files in os.walk(LOCAL_PATH):
    for file in files:
        if file == 'dataset_info.json':
            info_files.append(os.path.join(root, file))
            print(f"Найден dataset_info.json: {os.path.relpath(os.path.join(root, file), LOCAL_PATH)}")

print("\n" + "="*60)
print("ВОССТАНОВЛЕНИЕ ДАТАСЕТА")
print("="*60)

In [ ]:
# ПРАВИЛЬНАЯ ЗАГРУЗКА ДАТАСЕТА
from datasets import Dataset, DatasetDict
from huggingface_hub import login
import os

# Авторизация
login(token="YOUR_HF_TOKEN")  # Замените на ваш токен

LOCAL_PATH = "/Users/polinakremneva/code/deep-pavlov/clarqa_ru_complete"
REPO_ID = "DeepPavlov/clarqa_ru"

print("="*60)
print("ЗАГРУЗКА ДАТАСЕТА ИЗ ЛОКАЛЬНОЙ ПАПКИ")
print("="*60)

# Создаём основной DatasetDict
dataset = DatasetDict()

# Загружаем single_turn как DatasetDict
print("\nЗагрузка single_turn...")
single_turn_path = os.path.join(LOCAL_PATH, "single_turn")
single_turn = DatasetDict.load_from_disk(single_turn_path)
print(f"   Загружен single_turn со сплитами: {list(single_turn.keys())}")
for split_name, ds in single_turn.items():
    print(f"      - {split_name}: {len(ds)} примеров")
    print(f"        Колонки: {ds.column_names}")
dataset["single_turn"] = single_turn

# Загружаем multi_turn как DatasetDict
print("\nЗагрузка multi_turn...")
multi_turn_path = os.path.join(LOCAL_PATH, "multi_turn")
multi_turn = DatasetDict.load_from_disk(multi_turn_path)
print(f"   Загружен multi_turn со сплитами: {list(multi_turn.keys())}")
for split_name, ds in multi_turn.items():
    print(f"      - {split_name}: {len(ds)} примеров")
    print(f"        Колонки: {ds.column_names}")
dataset["multi_turn"] = multi_turn

print("\n" + "="*60)
print("ПРОВЕРКА ДАННЫХ")
print("="*60)

# Проверяем наличие русских переводов
for config_name, ds_dict in dataset.items():
    print(f"\n{config_name}:")
    for split_name, ds in ds_dict.items():
        print(f"   {split_name}: {len(ds)} примеров")
        
        # Проверяем первый пример
        example = ds[0]
        print(f"      Колонки: {list(example.keys())}")
        
        if 'context_ru' in example:
            context_ru = example['context_ru']
            if isinstance(context_ru, list):
                print(f"      context_ru (первый диалог): {context_ru[0][:80]}...")
            else:
                print(f"      context_ru: {context_ru[:80]}...")
        
        if 'entity1_ru' in example:
            entity1_ru = example['entity1_ru']
            if isinstance(entity1_ru, str) and len(entity1_ru) > 80:
                print(f"      entity1_ru: {entity1_ru[:80]}...")
            else:
                print(f"      entity1_ru: {entity1_ru}")
        
        break  # Показываем только первый сплит для каждого конфига

print("\n" + "="*60)
print(f"ЗАГРУЗКА НА HUGGING FACE HUB")
print(f"Репозиторий: {REPO_ID}")
print("="*60)

# Загружаем датасет на Hub
try:
    dataset.push_to_hub(
        REPO_ID,
        private=False,
        commit_message="Upload translated CLARQA dataset (EN -> RU) with Russian translations for context, entity1, and entity2"
    )
    
    print(f"\nУСПЕХ! Датасет успешно загружен!")
    print(f"   URL: https://huggingface.co/datasets/{REPO_ID}")
    
except Exception as e:
    print(f"\nОшибка при загрузке: {e}")
    print("\nПробуем альтернативный способ - загружаем конфигурации по отдельности...")
    
    # Альтернативный способ: загружаем каждую конфигурацию отдельно
    for config_name, ds_dict in dataset.items():
        print(f"\nЗагрузка конфигурации: {config_name}")
        try:
            ds_dict.push_to_hub(
                REPO_ID,
                config_name=config_name,
                private=False,
                commit_message=f"Upload {config_name} config with Russian translations"
            )
            print(f"   {config_name} успешно загружен")
        except Exception as ex:
            print(f"   Ошибка загрузки {config_name}: {ex}")

print("\n" + "="*60)
print("ПРОВЕРКА ЗАГРУЖЕННОГО ДАТАСЕТА")
print("="*60)

# Проверяем, что датасет загрузился правильно
from datasets import load_dataset, get_dataset_config_names

try:
    # Получаем список конфигураций
    configs = get_dataset_config_names(REPO_ID)
    print(f"\nДоступные конфигурации: {configs}")
    
    # Проверяем single_turn
    if 'single_turn' in configs:
        print("\nПроверка single_turn...")
        ds_test = load_dataset(REPO_ID, 'single_turn', split='train')
        print(f"   Загружено {len(ds_test)} примеров")
        print(f"   Пример:")
        print(f"      context_ru: {ds_test[0]['context_ru'][:100]}...")
        print(f"      entity1_ru: {ds_test[0]['entity1_ru'][:100]}...")
        print(f"      entity2_ru: {ds_test[0]['entity2_ru'][:100]}...")
        print(f"      label: {ds_test[0]['label']}")
    
    # Проверяем multi_turn
    if 'multi_turn' in configs:
        print("\nПроверка multi_turn...")
        ds_test = load_dataset(REPO_ID, 'multi_turn', split='train')
        print(f"   Загружено {len(ds_test)} примеров")
        print(f"   Пример (первый диалог):")
        print(f"      context_ru[0]: {ds_test[0]['context_ru'][0][:100]}...")
        print(f"      entity1_ru: {ds_test[0]['entity1_ru'][:100]}...")
        print(f"      label: {ds_test[0]['label']}")
    
    print("\n" + "="*60)
    print("🎉 ВСЕ ГОТОВО! Датасет доступен для использования")
    print("="*60)
    print(f"\nИспользование датасета:")
    print(f"```python")
    print(f"from datasets import load_dataset")
    print(f"")
    print(f"# Загрузка single_turn")
    print(f"dataset = load_dataset('{REPO_ID}', 'single_turn')")
    print(f"train_data = dataset['train']")
    print(f"")
    print(f"# Загрузка multi_turn")
    print(f"dataset = load_dataset('{REPO_ID}', 'multi_turn')")
    print(f"train_data = dataset['train']")
    print(f"```")
    
except Exception as e:
    print(f"\nНе удалось проверить загрузку: {e}")
    print("Проверьте датасет вручную по ссылке выше")